# M/M/2 (max 200 m)

Twenty vessel arrivals, two service slots and a shared 200 m quay. The only Excel input is the **info** worksheet; all visits are generated below. The data worksheet is not read.

Customers (vessels) and the service station (quay) are composed of mix-ins, in the same way as objects in OpenTNSim.

In [1]:
# package(s) for data handling
import pandas as pd
from IPython.display import display

# OpenQTSim mixins and utils
import openqtsim
from openqtsim.core import Identifiable, HasResource, HasLength, ExtraMetadata
from openqtsim.core import HasArrivalSchedule, HasHandlingTimes, HasMaxWait
from openqtsim.core import create_object, inspect_object
from openqtsim.visualizations import plot_gantt

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

This notebook is executed with OpenQTSim version v0.5.1


#### 0. Define input

Use **exactly one** stopping limit: `nr_arrivals=20` and `max_duration_days=None`, or `nr_arrivals=None` and e.g. `max_duration_days=60`. A duration limit includes arrivals at the boundary; service and waiting then drain to completion or reneging, even beyond that limit.

`D` uses fixed times. `M` samples independent negative-exponential times. Stochastic first arrivals have mean `AT`; subsequent return intervals have mean `IAT`, separately for each vessel. Stochastic service has vessel-specific mean `loading + unloading`. Arrival and service seeds must be different; changing the service seed does not change the generated arrivals.

These are vessel-specific renewal streams and service mixtures, not necessarily a homogeneous textbook M/M/2 queue. Samples are generated in advance and replayed through the simulator's deterministic-table interface to preserve vessel identity.

In [ ]:
example_title = "M M 2 (max 200 m)"

nr_arrivals = 20
max_duration_days = None

arrival_distribution = "M"
service_distribution = "M"
arrival_seed = 42
service_seed = 43

#### 1. Create service station
Next we create a Quay-object (our service station) out of the available mix-ins:

In [3]:
# make your preferred Quay class out of available mix-ins.
Quay = create_object(
    "Quay",
    (
        Identifiable,  # allows to give the object a name and a random ID
        HasResource,   # number of service slots (c in Kendall's notation)
        HasLength,     # quay length that is shared by the vessels at the quay
    ),
)

We can inspect the object to determine which inputs are required and optional:

In [4]:
inspect_object(Quay, show_parameter_table=True)

,parameter,class,default,status
0,length,HasLength,,mandatory
1,name,Identifiable,,mandatory
2,nr_resources,HasResource,1,optional (unused)
3,id,Identifiable,None,optional (unused)


We define the required input and create the quay object. A vessel is only admitted when both a service slot and enough remaining quay length are available:

In [5]:
data_quay = {
    "name": "Quay",       # required by Identifiable
    "nr_resources": 2,    # optional for HasResource: two service slots
    "length": 200,        # required by HasLength: 200 m quay length
}

quay = Quay(**data_quay)

#### 2. Create vessels

All times in `info` are **days**, and lengths are metres. `AT` is the first arrival time (or its mean when stochastic); `IAT` is each vessel's return interval (or its mean). Service time is calculated from `loading + unloading`, not the redundant `ST` column. The generated simulation tables use **hours**.

In [6]:
info = pd.read_excel('1002 - D D 2 (max 200 m).xlsx', sheet_name='info')
info

,name,IAT,ST,AT,length,maxwait,unloading,loading
0,Yellowstar,7,3,2,80,1,2,1
1,ABC,30,7,12,100,2,4,3
2,Oil Trade,18,2,5,100,1,2,0
3,Gen. Cargo Trade,60,4,24,160,2,3,1
4,Fruit Trade,10,6,9,90,0,3,3
5,United,16,5,2,80,2,1,4


We make a Vessel class out of the available mix-ins:

In [7]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel",
    (
        Identifiable,        # allows to give the object a name and a random ID
        HasLength,           # vessel length (m) that occupies the quay
        HasArrivalSchedule,  # first arrival AT and return interval IAT (days)
        HasHandlingTimes,    # unloading and loading time (days); service time is their sum
        HasMaxWait,          # maximum waiting time (days) before the vessel reneges
        ExtraMetadata,       # stores all other inputs (here the redundant ST column) as metadata
    ),
)

We can inspect the object to determine which inputs are required and optional:

In [8]:
inspect_object(Vessel, show_parameter_table=True)

,parameter,class,default,status
0,AT,HasArrivalSchedule,,mandatory
1,IAT,HasArrivalSchedule,,mandatory
2,length,HasLength,,mandatory
3,maxwait,HasMaxWait,,mandatory
4,name,Identifiable,,mandatory
5,loading,HasHandlingTimes,0,optional (unused)
6,unloading,HasHandlingTimes,0,optional (unused)
7,id,Identifiable,None,optional (unused)


Each row of the info sheet is a data dict for one vessel object. The redundant `ST` column ends up in `vessel.metadata`:

In [9]:
vessels = [Vessel(**data_vessel) for data_vessel in info.to_dict('records')]

pd.DataFrame([{
    "name": vessel.name, "length": vessel.length, "AT": vessel.AT, "IAT": vessel.IAT,
    "service_time": vessel.service_time, "maxwait": vessel.maxwait, "metadata": vessel.metadata,
} for vessel in vessels])

,name,length,AT,IAT,service_time,maxwait,metadata
0,Yellowstar,80,2,7,3,1,{'ST': 3}
1,ABC,100,12,30,7,2,{'ST': 7}
2,Oil Trade,100,5,18,2,1,{'ST': 2}
3,Gen. Cargo Trade,160,24,60,4,2,{'ST': 4}
4,Fruit Trade,90,9,10,6,0,{'ST': 6}
5,United,80,2,16,5,2,{'ST': 5}


#### 3. Generate the arrival and service tables

Each row represents one visit. Visits are ordered by arrival time, with ties resolved by the original info-sheet row order. The simulator pairs both tables by row position. Subsequent visits remain scheduled even if an earlier visit reneges. `maxwait` is converted from days to hours, without randomisation.

In [10]:
arrival_table, service_table = openqtsim.generate_vessel_tables(
    vessels,
    nr_arrivals=nr_arrivals,
    max_duration_days=max_duration_days,
    arrival_distribution=arrival_distribution,
    service_distribution=service_distribution,
    arrival_seed=arrival_seed,
    service_seed=service_seed,
)
print(f'Generated {len(arrival_table)} vessel arrivals; all table times are hours.')
display(arrival_table)
display(service_table)

Generated 20 vessel arrivals; all table times are hours.


,name,vessel_id,visit,IAT,AT,length,maxwait
0,United,6,1,8.163214,8.163214,80,48
1,United,6,2,19.673987,27.837201,80,48
2,Yellowstar,1,1,18.664126,46.501327,80,24
3,Oil Trade,3,1,8.396336,54.897663,100,24
4,United,6,3,31.999681,86.897344,80,48
5,United,6,4,22.386715,109.284059,80,48
6,Oil Trade,3,2,17.615306,126.899365,100,24
7,Gen. Cargo Trade,4,1,32.804554,159.703919,160,48
8,Oil Trade,3,3,26.157198,185.861117,100,24
9,Oil Trade,3,4,17.361303,203.222420,100,24


,name,ST
0,United,32.043071
1,United,236.379178
2,Yellowstar,24.425042
3,Oil Trade,77.639549
4,United,64.961633
5,United,26.108826
6,Oil Trade,6.946678
7,Gen. Cargo Trade,27.761872
8,Oil Trade,18.481896
9,Oil Trade,24.880551


#### 4. Run simulation

Admission requires both a free service slot and enough remaining length. Strict FIFO prevents bypassing a blocked vessel. A vessel that cannot start by `AT + maxwait` reneges and leaves the queue permanently. Service starting exactly at the deadline is allowed; zero patience allows immediate service only. The quay object sets the number of servers and the quay length of the queue.

In [11]:
if arrival_table.empty:
    raise ValueError('No arrivals within the selected duration; increase the limit.')
A = openqtsim.ArrivalProcess('D', arrival_table)
S = openqtsim.ServiceProcess('D', service_table)
q = openqtsim.Queue(A, S, station=quay)
sim = openqtsim.Simulation(q)
sim.run(len(arrival_table))
summary = sim.get_stats()

Departures: 15 served, 5 reneged
Waiting time in units of service time: 0.1929
Observation horizon: 0 to 1231.0290 hours

Rho_system: system utilisation: 0.6573
Rho_server: server utilisation: 0.4847
P_0: probability nobody in the system: 0.3427

L_s: average nr of customers in the system: 1.2188
L_q: average nr of customers in the queue: 0.2493
W_s: average time spent in the system: 75.0170
W_q: average time spent in the queue: 15.3474

IAT: average inter arrival time: 57.7533
ST: average service time: 79.5595



In [12]:
sim.plot_system_state()

#### 5. Inspect output

In [13]:
df, df_sys = sim.return_log()
display(df)
display(df_sys.head(10))

,c_id,name,length,maxwait,status,departure,IAT,ST,AT,TSB,TSE,TCSS,TCWQ,ITS,s_id
0,1,United,80,48,served,40.206285,8.163214,32.043071,8.163214,8.163214,40.206285,32.043071,0.000000,8.163214,1.0
9,2,United,80,48,served,264.216379,19.673987,236.379178,27.837201,27.837201,264.216379,236.379178,0.000000,27.837201,2.0
1,3,Yellowstar,80,24,served,70.926369,18.664126,24.425042,46.501327,46.501327,70.926369,24.425042,0.000000,6.295042,1.0
3,4,Oil Trade,100,24,served,148.565918,8.396336,77.639549,54.897663,70.926369,148.565918,93.668255,16.028706,0.000000,1.0
2,5,United,80,48,reneged,134.897344,31.999681,NaN,86.897344,NaN,NaN,48.000000,48.000000,NaN,NaN
5,6,United,80,48,served,174.674744,22.386715,26.108826,109.284059,148.565918,174.674744,65.390685,39.281859,0.000000,1.0
4,7,Oil Trade,100,24,reneged,150.899364,17.615306,NaN,126.899364,NaN,NaN,24.000000,24.000000,NaN,NaN
6,8,Gen. Cargo Trade,160,48,reneged,207.703918,32.804554,NaN,159.703918,NaN,NaN,48.000000,48.000000,NaN,NaN
7,9,Oil Trade,100,24,served,226.185815,26.157198,18.481896,185.861117,207.703918,226.185815,40.324698,21.842802,33.029175,1.0
8,10,Oil Trade,100,24,served,251.066366,17.361303,24.880551,203.222420,226.185815,251.066366,47.843946,22.963395,0.000000,1.0


,t,c_s,c_q
0,0.000000,0,0
1,8.163214,1,1
2,8.163214,1,0
3,27.837201,2,1
4,27.837201,2,0
5,40.206285,1,0
6,46.501327,2,1
7,46.501327,2,0
8,54.897663,3,1
9,70.926369,2,1


#### 6. Gantt chart: arrival, waiting, service and reneging

One row per vessel visit, labelled by vessel name. Orange is waiting, blue is actual service, a green cross marks departure after service, and a **red cross** marks reneging. Reneged visits have no service bar. Hover labels show at most two decimals in hours, with days in parentheses. The table below includes status and actual departure time.

In [14]:
fig = plot_gantt(df, title=f"{example_title}: {len(df)} vessel arrivals")
fig.show()
df.sort_values(["AT", "c_id"])[["c_id", "name", "length", "maxwait", "status", "AT", "TSB", "TSE",
                                "departure", "TCWQ", "ST", "TCSS", "s_id"]]

,c_id,name,length,maxwait,status,AT,TSB,TSE,departure,TCWQ,ST,TCSS,s_id
0,1,United,80,48,served,8.163214,8.163214,40.206285,40.206285,0.000000,32.043071,32.043071,1.0
9,2,United,80,48,served,27.837201,27.837201,264.216379,264.216379,0.000000,236.379178,236.379178,2.0
1,3,Yellowstar,80,24,served,46.501327,46.501327,70.926369,70.926369,0.000000,24.425042,24.425042,1.0
3,4,Oil Trade,100,24,served,54.897663,70.926369,148.565918,148.565918,16.028706,77.639549,93.668255,1.0
2,5,United,80,48,reneged,86.897344,NaN,NaN,134.897344,48.000000,NaN,48.000000,NaN
5,6,United,80,48,served,109.284059,148.565918,174.674744,174.674744,39.281859,26.108826,65.390685,1.0
4,7,Oil Trade,100,24,reneged,126.899364,NaN,NaN,150.899364,24.000000,NaN,24.000000,NaN
6,8,Gen. Cargo Trade,160,48,reneged,159.703918,NaN,NaN,207.703918,48.000000,NaN,48.000000,NaN
7,9,Oil Trade,100,24,served,185.861117,207.703918,226.185815,226.185815,21.842802,18.481896,40.324698,1.0
8,10,Oil Trade,100,24,served,203.222420,226.185815,251.066366,251.066366,22.963395,24.880551,47.843946,1.0
